[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/Escher_class/iJO1366_Escher_flux_exercise.ipynb)

# iJO1366 Flux Balance Analysis and Escher Visualization

Hands-on exercise with the *E. coli* genome-scale model **iJO1366** (BiGG). You will set growth media, run FBA, compare conditions, export fluxes for **Escher**, and simulate reaction knockouts.

---

## Exercises

1. Load iJO1366 and inspect the objective.
2. Simulate glucose aerobic vs anaerobic growth.
3. Simulate glycerol aerobic vs anaerobic growth.
4. Compare growth rates and explain differences.
5. Visualize fluxes in Escher.
6. Run reaction knockouts (PFK, G6PDH2r, CS, ATPS4rpp) under glucose aerobic medium.
7. Identify reactions with the largest flux changes.

## Setup

In [ ]:
import sys
!apt-get -qq install -y swig libgmp-dev 2>/dev/null
!{sys.executable} -m pip install -q cobra escher pandas numpy swiglpk ipywidgets

In [ ]:
import zipfile
from pathlib import Path

import cobra
from cobra.io import load_json_model
import escher
import pandas as pd

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

for folder in ['data', 'outputs', 'escher_outputs']:
    Path(folder).mkdir(parents=True, exist_ok=True)

MODEL_PATH = 'data/iJO1366.json'
MAP_PATH = 'data/iJO1366.Central metabolism.json'

!wget -q -O {MODEL_PATH} http://bigg.ucsd.edu/static/models/iJO1366.json
!wget -q -O "{MAP_PATH}" "http://bigg.ucsd.edu/escher_map_json/iJO1366.Central%20metabolism"

## Exercise 1 — Load model and inspect objective

Load the JSON model with COBRApy and inspect its size and growth objective.

In [ ]:
model = load_json_model(MODEL_PATH)

print(model)
print(f'Reactions:   {len(model.reactions)}')
print(f'Metabolites: {len(model.metabolites)}')
print(f'Genes:       {len(model.genes)}')
print(f'Objective:   {model.objective}')

## Medium definition

Set uptake bounds on exchange reactions inside `with model:` so changes are reverted after each simulation.

In [ ]:
CARBON = {'glucose': 'EX_glc__D_e', 'glycerol': 'EX_glyc_e'}
MUTANTS = ['PFK', 'G6PDH2r', 'CS', 'ATPS4rpp']


def set_medium(model, carbon_source, carbon_uptake=10, oxygen_uptake=20):
    """Set carbon and oxygen uptake bounds; close other defined carbon exchanges."""
    for ex_id in CARBON.values():
        if model.reactions.has_id(ex_id):
            model.reactions.get_by_id(ex_id).lower_bound = 0
    if model.reactions.has_id(carbon_source):
        model.reactions.get_by_id(carbon_source).lower_bound = -carbon_uptake
    if model.reactions.has_id('EX_o2_e'):
        model.reactions.get_by_id('EX_o2_e').lower_bound = (
            -oxygen_uptake if oxygen_uptake > 0 else 0
        )

## Exercises 2 & 3 — Wild-type simulations

Run FBA for glucose/glycerol under aerobic and anaerobic conditions.

In [ ]:
conditions = [
    ('glucose_aerobic',    CARBON['glucose'],  10, 20),
    ('glucose_anaerobic',  CARBON['glucose'],  10,  0),
    ('glycerol_aerobic',   CARBON['glycerol'], 10, 20),
    ('glycerol_anaerobic', CARBON['glycerol'], 10,  0),
]

fluxes = {}
summary = []

for name, carbon, c_uptake, o2 in conditions:
    with model:
        set_medium(model, carbon, carbon_uptake=c_uptake, oxygen_uptake=o2)
        solution = model.optimize()
        fluxes[name] = solution.fluxes
        summary.append({
            'condition': name,
            'status': solution.status,
            'objective_value': solution.objective_value,
            'carbon_source': carbon,
            'oxygen_uptake': o2,
            'knockout': None,
        })

growth_summary = pd.DataFrame(summary)
display(growth_summary)

## Exercise 6 — Reaction knockouts (glucose aerobic)

Knock out reactions with `reaction.knock_out()` inside the model context manager.

In [ ]:
for rxn_id in MUTANTS:
    if not model.reactions.has_id(rxn_id):
        continue
    name = f'mutant_{rxn_id}'
    with model:
        set_medium(model, CARBON['glucose'], carbon_uptake=10, oxygen_uptake=20)
        model.reactions.get_by_id(rxn_id).knock_out()
        solution = model.optimize()
        fluxes[name] = solution.fluxes
        summary.append({
            'condition': name,
            'status': solution.status,
            'objective_value': solution.objective_value,
            'carbon_source': CARBON['glucose'],
            'oxygen_uptake': 20,
            'knockout': rxn_id,
        })

growth_summary = pd.DataFrame(summary)
display(growth_summary)

## Export results for Escher

In [ ]:
def escher_csv(df):
    """Escher expects the first column to be named ID."""
    out = df.reset_index().rename(columns={'reaction': 'ID'})
    if out.columns[0] != 'ID':
        out = out.rename(columns={out.columns[0]: 'ID'})
    return out

growth_summary.to_csv('outputs/growth_summary.csv', index=False)

flux_table = pd.DataFrame(fluxes)
flux_table.index.name = 'reaction'

escher_csv(flux_table).to_csv('outputs/escher_reaction_fluxes_wide.csv', index=False)

for cols, fname in [
    (['glucose_aerobic', 'glucose_anaerobic'], 'escher_glucose_aerobic_vs_anaerobic.csv'),
    (['glucose_aerobic', 'glycerol_aerobic'], 'escher_glucose_vs_glycerol_aerobic.csv'),
]:
    escher_csv(flux_table[cols]).to_csv(f'outputs/{fname}', index=False)

mutant_cols = ['glucose_aerobic'] + [c for c in flux_table.columns if c.startswith('mutant_')]
escher_csv(flux_table[mutant_cols]).to_csv('outputs/escher_mutants_glucose_aerobic.csv', index=False)

## Exercises 4 & 7 — Flux analysis

In [ ]:
comparisons = {
    'glucose_aerobic_minus_anaerobic': ('glucose_aerobic', 'glucose_anaerobic'),
    'glucose_aerobic_minus_glycerol_aerobic': ('glucose_aerobic', 'glycerol_aerobic'),
}
for rxn_id in MUTANTS:
    col = f'mutant_{rxn_id}'
    if col in flux_table.columns:
        comparisons[f'glucose_aerobic_minus_{col}'] = ('glucose_aerobic', col)

top_rows = []
for label, (a, b) in comparisons.items():
    diff = flux_table[a] - flux_table[b]
    top = diff.abs().sort_values(ascending=False).head(20)
    for rank, rxn in enumerate(top.index, start=1):
        top_rows.append({
            'comparison': label,
            'rank': rank,
            'reaction': rxn,
            'flux_difference': diff[rxn],
            'abs_flux_difference': top[rxn],
        })

top_flux_changes = pd.DataFrame(top_rows)
top_flux_changes.to_csv('outputs/top_flux_changes.csv', index=False)
display(top_flux_changes.groupby('comparison').head(5))

## Exercise 5 — Visualize fluxes in Escher

Escher runs as a Jupyter widget in the notebook. The map below shows `glucose_aerobic` fluxes. Change `builder.reaction_data` to another condition to compare.

In [ ]:
builder = escher.Builder(
    model_json=MODEL_PATH,
    map_json=MAP_PATH,
    reaction_data=fluxes['glucose_aerobic'],
    height=600,
    menu='zoom',
    scroll_behavior='zoom',
    hide_secondary_metabolites=True,
)
builder

In [ ]:
# optional: save the current map for viewing outside the notebook
builder.save_html('escher_outputs/iJO1366_flux_visualization.html')

# try another condition, e.g. anaerobic:
# builder.reaction_data = fluxes['glucose_anaerobic']

## Download outputs

In [ ]:
ZIP_NAME = 'iJO1366_escher_exercise_outputs.zip'
if Path(ZIP_NAME).exists():
    Path(ZIP_NAME).unlink()

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['outputs', 'escher_outputs', 'data']:
        for path in Path(folder).rglob('*'):
            if path.is_file():
                zf.write(path, arcname=str(path))

---

## Loading CSV flux data at escher.github.io

The in-notebook map above is the main visualization. To use the exported CSV files in the web app:

1. Open [https://escher.github.io/](https://escher.github.io/)
2. **Model → Load COBRA Model** → `data/iJO1366.json`
3. **Map → Load Map JSON** → `data/iJO1366.Central metabolism.json`
4. **Data → Load reaction data** → one CSV from `outputs/` (first column must be `ID`)

Files with two flux columns let Escher compare conditions side by side.